# Treasure Trail — SBC Kids' Bible SLM
### QLoRA fine-tune + base-vs-tuned eval

**First: Runtime → Change runtime type → T4 GPU** (free T4 is enough), then **Runtime → Run all**.

Flow: clone repo → install → **baseline eval** (show the base *flattens/caves*) → **QLoRA fine-tune** on the 1,095-record dataset → **tuned eval** → the **base-vs-tuned results table**.

You'll need an **Anthropic API key** — used only by the eval judge (training doesn't need it). Cell 2 will prompt for it.

In [ ]:
!git clone https://github.com/graceyan212/bible-slm.git
%cd bible-slm
!pip install -q unsloth anthropic

## 1 · Config + judge key
Default base is Qwen3-4B. If the T4 runs low on memory or too slow, switch to the 1.7B line (and it's used consistently for train + eval).

In [ ]:
import os, getpass

MODEL = "unsloth/Qwen3-4B-Instruct-2507"      # faster/lighter option:
# MODEL = "unsloth/Qwen3-1.7B-Instruct"
os.environ["BASE_MODEL"] = MODEL               # train_qlora.py reads this

# Judge key (eval only). getpass so it never gets saved in the notebook.
os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your Anthropic API key: ")
# os.environ["JUDGE_MODEL"] = "claude-sonnet-5"   # uncomment if the judge errors on the default
print("Base model:", MODEL)

## 2 · Baseline eval — run BEFORE training
This proves the delta target exists: expect the base model to **flatten** on baptism / eternal-security and **cave** under pushback. (~52 scenarios + judge calls.)

In [ ]:
!python eval/run_eval.py --model base --hf {MODEL} --out results_base.json

## 3 · Fine-tune (QLoRA, ~30–60 min)
Trains on `data/train_v2.jsonl` (1,095 verified records; prints the data hash) → writes the adapter to `./sbc-lora`.

In [ ]:
!python train/train_qlora.py

## 4 · Tuned eval — same 52 scenarios, same system prompt

In [ ]:
!python eval/run_eval.py --model tuned --hf {MODEL} --adapter ./sbc-lora --out results_tuned.json

## 5 · Results table — base vs tuned (the headline artifact)

In [ ]:
!python eval/run_eval.py --compare results_base.json results_tuned.json --md results_table.md
from IPython.display import Markdown, display
display(Markdown(open('results_table.md').read()))

**A win =** tuned beats base on **demo FLATTEN rate** + **hold-under-pressure**, *without* open OVER_HOLD / deflect-leak rising or safe_core task-quality regressing. That's the "behavior from data" proof.

Colab wipes when the runtime ends — the next cell downloads your results. Send `results_table.md` back and I'll build the demo.

In [ ]:
from google.colab import files
for f in ['results_table.md', 'results_base.json', 'results_tuned.json']:
    files.download(f)